# LLM Reasoning Benchmark Suite: Qwen2.5-Coder-7B-Instruct

Notebook thực hiện quy trình đánh giá chuẩn hóa năng lực suy luận toán học trên hai tập dữ liệu **MATH-500** và **GSM8K** với 4 phương pháp:

1. **Direct (Baseline)**: Zero-shot Direct Answering.
2. **CoT (Baseline)**: Chain-of-Thought Step-by-Step Natural Language Reasoning (Wei et al., NeurIPS 2022).
3. **SymCode (Baseline)**: Neurosymbolic Equation Solving với SymPy (ACL 2026) bản tiêu chuẩn.
4. **SymPlan (Ours - Đề xuất)**: Verifiable SymPlanner IR Pipeline (Structured IR -> Normalization -> Codegen -> Bidirectional Relation Verifier -> Targeted Code Repair).

---
**Đặc điểm nổi bật**:
- Hỗ trợ lọc theo độ khó Level 1–5 cho cả MATH-500 và GSM8K.
- Cơ chế **Auto-Resume** tự động lưu và khôi phục tiến trình tại `/kaggle/working/` khi bị ngắt kết nối session.
- Báo cáo kết quả đa chiều: Bảng Overall, Bảng Subject x Difficulty, và Bảng Chẩn đoán độ tin cậy IR Diagnostics.

In [ ]:
# 1. Khởi tạo môi trường, định vị thư mục mã nguồn và nạp vào sys.path
import os
import sys
import glob

# Tự động tìm thư mục chứa run_notebook.py trên Kaggle Input hoặc Workspace
for search_root in ["/kaggle/input", ".", ".."]:
    if os.path.exists(search_root):
        matches = glob.glob(f"{search_root}/**/run_notebook.py", recursive=True)
        if matches:
            src_dir = os.path.abspath(os.path.dirname(matches[0]))
            if src_dir not in sys.path:
                sys.path.insert(0, src_dir)
            break

from run_notebook import setup_environment

setup_environment()

In [ ]:
# 2. BẢNG ĐIỀU KHIỂN CẤU HÌNH BENCHMARK (Tùy chỉnh tại đây)
DATASET = "math500"                                 # Chọn "math500" hoặc "gsm8k"
METHODS = ["Direct", "CoT", "SymCode", "SymPlan"]  # Baselines: Direct, CoT, SymCode | Ours: SymPlan
NUM_SAMPLES = 5                                     # Số lượng mẫu (None = toàn bộ tập dữ liệu)
FILTER_LEVELS = [1, 2, 3]                           # Lọc độ khó [1, 2, 3], [4, 5] hoặc None (tất cả)

from run_notebook import build_config

config = build_config(
    dataset_name=DATASET,
    methods=METHODS,
    num_samples=NUM_SAMPLES,
    filter_levels=FILTER_LEVELS
)

In [ ]:
# 3. Thực thi Benchmark & Hiển thị kết quả đa chiều
from run_notebook import run_benchmark_pipeline

results = run_benchmark_pipeline(config)